# Monitor — AVIDa-hIL6 acquisition

Downloads **AVIDa-hIL6** into `data/raw/` and reduces it to the four antigens of
the working set. Nothing else: preprocessing (expression-wrapper stripping,
mutation parsing, flip-set extraction) happens downstream.

Source: `COGNANO/AVIDa-hIL6` on HuggingFace (DOI 10.57967/hf/4241),
paper arXiv:2306.03329 (NeurIPS 2023 D&B), code github.com/cognano/AVIDa-hIL6.
**Licence CC BY-NC 4.0** — non-commercial only; state this in any writeup.

Plain HTTPS, no token and no `huggingface_hub` dependency. The documented
`load_dataset("COGNANO/AVIDa-hIL6")` route raises `DatasetGenerationCastError`:
the two CSVs have mismatched schemas and are not declared as separate configs.

| file | size | contents |
|---|---|---|
| `AVIDa-hIL6.csv` | 104.7 MB | 573,891 VHH/antigen pairs, binary labels |
| `antigen_sequences.csv` | 7.1 kB | 31 antigens: IL-6 WT + 30 point mutants |

> Note: sequences in both files carry expression wrappers (VHH: 22-residue
> phagemid signal peptide + His6; antigen: IL-6 precursor signal peptide
> P05231 1–29 + His6). They are left intact here — strip them before any
> structure prediction or MD.

## 0. Configuration

In [1]:
import hashlib
import urllib.request
from pathlib import Path

import pandas as pd

WD  = Path("/Users/roessner/Documents/PostDoc/Data/261003_MD_single_interface_mutations")
RAW = WD / "data" / "raw"
RAW.mkdir(parents=True, exist_ok=True)

REPO = "https://huggingface.co/datasets/COGNANO/AVIDa-hIL6/resolve/main"
FILES = {
    "AVIDa-hIL6.csv":        104_655_792,
    "antigen_sequences.csv":       7_133,
}

## 1. Download

Streamed, with a size check so a re-run is a no-op. A file present at the wrong
size is re-fetched — a truncated 105 MB CSV otherwise fails much later and much
less obviously.

In [2]:
def fetch(name, expect_bytes, chunk=1 << 20):
    dest = RAW / name
    if dest.exists() and dest.stat().st_size == expect_bytes:
        print(f"{name:24s} cached  ({dest.stat().st_size:,} B)")
        return dest
    if dest.exists():
        print(f"{name:24s} size mismatch "
              f"({dest.stat().st_size:,} != {expect_bytes:,}) -> refetching")
    tmp, got = dest.with_suffix(dest.suffix + ".part"), 0
    with urllib.request.urlopen(f"{REPO}/{name}", timeout=120) as r, open(tmp, "wb") as fh:
        while True:
            buf = r.read(chunk)
            if not buf:
                break
            fh.write(buf)
            got += len(buf)
            if got % (1 << 24) < chunk:
                print(f"  {name} {got/1e6:7.1f} / {expect_bytes/1e6:.1f} MB")
    print(f"{name:24s} done    ({got:,} B)")
    assert got == expect_bytes, f"{name}: got {got:,} B, expected {expect_bytes:,} B"
    tmp.rename(dest)
    return dest


paths = {n: fetch(n, sz) for n, sz in FILES.items()}

for n, p in paths.items():
    print(f"{n:24s} sha256 {hashlib.sha256(p.read_bytes()).hexdigest()}")

AVIDa-hIL6.csv           cached  (104,655,792 B)
antigen_sequences.csv    cached  (7,133 B)
AVIDa-hIL6.csv           sha256 917d90f111d759edc3440f577458d42ef89ff0315b186d6c266c120143b20cde
antigen_sequences.csv    sha256 8f023581932e626ba1e2da4f7f861b998b8fcb7516aedb785d831f8c36984e68


## 2. Reduce to the working set

Four antigens: WT IL-6 plus the three alanine mutants chosen as the FMD panel.
Each sits at a receptor interface in 1P9M (biological assembly) and is
solvent-exposed, so a loss of VHH binding there is plausibly a contact effect,
not a folding effect.

| antigen | mature | binds | partner domain | site | buried |
|---|---|---|---|---|---|
| `IL-6_F102A` | F73  | IL-6Ra | D2/D3 (CBM) | I  | 71 A^2 |
| `IL-6_E138A` | E109 | gp130  | D2 (CBM)    | II | 82 A^2 |
| `IL-6_F153A` | F124 | gp130  | D2 (CBM)    | II | 33 A^2 |

`IL-6_F102A` is additionally contacted by three camelid binders (VHH6 in 5FUC,
Fab 61H7 in 4O9H, Fab 68F2 in 4ZS7), making it the best-evidenced position in
the whole 30-position scan.

Mutation labels are in **precursor** numbering (the frame of the `Ag_sequence`
strings); mature = label - 29.

In [3]:
pairs = pd.read_csv(paths["AVIDa-hIL6.csv"])
ags   = pd.read_csv(paths["antigen_sequences.csv"])
print(f"loaded {len(pairs):,} pairs, {len(ags)} antigens")

TARGETS = ["IL-6_WTs", "IL-6_F102A", "IL-6_E138A", "IL-6_F153A"]

missing = set(TARGETS) - set(ags.Ag_label)
assert not missing, f"target antigens absent from the release: {missing}"

sub = pairs[pairs.Ag_label.isin(TARGETS)].copy()

counts = (sub.groupby("Ag_label")
             .agg(pairs=("label", "size"),
                  binders=("label", "sum"),
                  unique_VHH=("VHH_sequence", "nunique"))
             .assign(non_binders=lambda d: d.pairs - d.binders,
                     pct_binder=lambda d: (d.binders / d.pairs * 100).round(2))
             .loc[TARGETS, ["pairs", "binders", "non_binders", "pct_binder", "unique_VHH"]])

print(counts.to_string())
print(f"\nTOTAL  pairs {len(sub):,}   binders {int(sub.label.sum()):,}   "
      f"non-binders {len(sub) - int(sub.label.sum()):,}   "
      f"({sub.label.mean():.2%} positive)")
print(f"unique VHH sequences across the four targets: {sub.VHH_sequence.nunique():,}")

# how many VHHs were assayed against how many of the four
cov = sub.groupby("VHH_sequence").Ag_label.nunique().value_counts().sort_index()
print("\nVHHs by number of the four targets they were tested on:")
print(cov.rename_axis("n_targets").to_frame("n_VHH").to_string())

PRO = WD / "data" / "processed"
PRO.mkdir(parents=True, exist_ok=True)
sub.to_csv(PRO / "subset_4targets.csv", index=False)
print(f"\nwrote {PRO / 'subset_4targets.csv'}  ({len(sub):,} rows)")

loaded 573,891 pairs, 31 antigens
            pairs  binders  non_binders  pct_binder  unique_VHH
Ag_label                                                       
IL-6_WTs    13020      540        12480        4.15       13020
IL-6_F102A  24040      615        23425        2.56       24040
IL-6_E138A  10974     1402         9572       12.78       10974
IL-6_F153A  11469     1158        10311       10.10       11469

TOTAL  pairs 59,503   binders 3,715   non-binders 55,788   (6.24% positive)
unique VHH sequences across the four targets: 26,776

VHHs by number of the four targets they were tested on:
           n_VHH
n_targets       
1          10817
2           5226
3           4698
4           6035

wrote /Users/roessner/Documents/PostDoc/Data/261003_MD_single_interface_mutations/data/processed/subset_4targets.csv  (59,503 rows)


## 3. Clonotype clustering (full repertoire)

Assigns every VHH in the release to a **clonotype** — a set of sequences
descended from one founder B cell. Two VHHs in the same clonotype are one
evolutionary experiment sampled twice, so they are *not* independent evidence
and must not be counted as separate systems downstream.

Run on the **whole release**, not on any subset: single-linkage cluster
membership depends on which sequences are present, so clustering a subset
inflates apparent independence. Computing it once here means every downstream
selection just joins on `clonotype`.

**Why not whole-sequence identity.** A VHH is a germline scaffold with three
hypervariable loops. Framework is ~85% of the domain and is near-identical
across *all* VHHs from one animal, related or not — measured on this repertoire,
random unrelated pairs are already **66% identical** (90th pct 74%). An 80%
whole-domain threshold therefore sits in the noise. CDR3 is the VDJ junction and
has no such floor: random pairs share **16%** (90th pct 30%), so an 80% CDR3
threshold is ~50 points clear of the null.

**Definition used here:** same CDR3 length + >= 80% CDR3 identity (positionwise,
no alignment needed since lengths match), single-linkage.

> Caveat: this is CDR3-only. The full clonotype definition also requires the same
> **V and J germline genes**, which needs ANARCI or IgBLAST with camelid
> (*Vicugna pacos*) germlines. Adding V/J can only *split* clonotypes, never merge
> them, so the counts here are a **lower bound** on independence.
>
> CDR3 is located by regex rather than IMGT numbering and parses ~85% of domains;
> the rest stay singletons. Good enough for counting independence, not for a
> figure.

In [4]:
import re

import numpy as np

VHH_SIGNAL = "MKYLLPTAAAGLLLLAAQPAMA"      # 22 aa phagemid signal peptide
CDR3_TH    = 0.80                          # CDR3 identity threshold

# Domain = raw sequence minus the fixed-length signal peptide and the His tag.
# Sliced by length, not prefix-matched: ~1.4% of entries carry point mutations
# inside the signal peptide and would otherwise keep the whole wrapper.
dom = (pd.Series(pairs.VHH_sequence.unique())
         .str[len(VHH_SIGNAL):]
         .str.replace(r"H+$", "", regex=True)
         .drop_duplicates()
         .reset_index(drop=True))
print(f"unique VHH domains in the release: {len(dom):,}")

_START = re.compile(r"[YFH][YFVAHL][CG]")   # end of FR3 (…YYC / …YFC / …HYG)
_END   = re.compile(r"WG.G")                # start of FR4 (WGQG / WGKG …)

def cdr3(s):
    a = list(_START.finditer(s)); b = list(_END.finditer(s))
    if not a or not b:
        return None
    i, j = a[-1].end(), b[-1].start()
    return s[i:j] if 2 < j - i < 45 else None

clo = pd.DataFrame({"VHH_domain": dom, "cdr3": dom.map(cdr3)})
print(f"CDR3 parsed: {clo.cdr3.notna().sum():,} ({clo.cdr3.notna().mean():.1%}) "
      f"— unparsed stay singletons")

# union-find over (equal CDR3 length) blocks, vectorised Hamming
parent = np.arange(len(clo))
def find(x):
    while parent[x] != x:
        parent[x] = parent[parent[x]]
        x = parent[x]
    return x

ok = clo[clo.cdr3.notna()]
for L, g in ok.groupby(ok.cdr3.str.len()):
    idx = g.index.to_numpy()
    M = np.frombuffer("".join(g.cdr3).encode(), dtype=np.uint8).reshape(len(g), L)
    need = int(np.ceil(CDR3_TH * L))
    step = max(1, int(4e7 // (len(g) * L)))
    for i in range(0, len(g), step):
        eq = (M[i:i + step, None, :] == M[None, :, :]).sum(2)
        for r, c in zip(*np.nonzero(eq >= need)):
            a, b = idx[i + r], idx[c]
            if a < b:
                ra, rb = find(a), find(b)
                if ra != rb:
                    parent[rb] = ra

clo["clonotype"] = [find(i) for i in range(len(clo))]
sizes = clo.clonotype.value_counts()
print(f"\n{len(clo):,} domains  ->  {clo.clonotype.nunique():,} clonotypes")
print(f"  singletons {int((sizes == 1).sum()):,}   largest {int(sizes.max()):,}")
print("\nclonotype size distribution (head):")
print(sizes.value_counts().sort_index().head(8)
           .rename_axis("size").to_frame("n_clonotypes").to_string())

clo.to_csv(PRO / "clonotypes.csv", index=False)
print(f"\nwrote {PRO / 'clonotypes.csv'}  ({len(clo):,} rows)")

unique VHH domains in the release: 38,109
CDR3 parsed: 32,195 (84.5%) — unparsed stay singletons



38,109 domains  ->  17,212 clonotypes
  singletons 14,723   largest 2,647

clonotype size distribution (head):
      n_clonotypes
size              
1            14723
2             1262
3              353
4              206
5              119
6               82
7               76
8               54

wrote /Users/roessner/Documents/PostDoc/Data/261003_MD_single_interface_mutations/data/processed/clonotypes.csv  (38,109 rows)


## 4. Flip-pair extraction

A **flip pair** is one VHH assayed against two antigens with *opposite* labels:

```
VHH #4821   IL-6_WTs  -> binder        IL-6_F102A -> non-binder
```

The VHH is byte-identical on both sides, so the only difference is a single
alanine substitution on the antigen. Whatever separates binder from non-binder
has to come from that one side chain — which is the causal handle the whole
benchmark rests on. Two useful consequences:

* **Balance for free.** Each pair contributes exactly one binder and one
  non-binder, despite the raw data being ~26:1.
* **No docking asymmetry.** The mutant complex is built by mutating one residue
  *in the WT bound pose*, so neither side needs a docking step that could
  correlate with the label.

**Order matters, and it is not clonotype-first.** Flips are defined on *exact*
VHH identity; clonotypes from section 3 are joined on afterwards, only to count
independence. Defining flips at clonotype level would compare *different* VHHs
and destroy the pairing.

Two directions are recorded. `lose` (binds WT, fails on the mutant) is the
mechanistically interpretable one. `gain` is reported but is a poor FMD target:
an alanine substitution *creating* a binder is not a mechanism these simulations
have reason to capture.

> The count that matters downstream is **independent clonotypes**, not pairs.
> Pairs within one clonotype are one lineage sampled repeatedly; any panel
> selection, bootstrap or CI must be taken over clonotypes.

In [5]:
WT = "IL-6_WTs"
MUTANTS = [t for t in TARGETS if t != WT]

# Domain-level identity: two raw sequences differing only in the signal peptide
# are the same binder, so collapse to the domain before pairing.
f = sub.copy()
f["VHH_domain"] = (f.VHH_sequence.str[len(VHH_SIGNAL):]
                    .str.replace(r"H+$", "", regex=True))

# Drop truncated domains. A complete VHH is ~111-132 aa (1st-99th pct of this
# release, median 124); 0.4% fall below 110 and are deletion/chimera artifacts
# from read assembly, missing CDR1 + FR2 + CDR2 entirely. They cannot fold into
# an Ig domain, so any structure predicted for them is fiction and MD on them is
# meaningless. Lower bound only: long domains are legitimate (camelid CDR3s run
# long, max here 151), it is the short tail that is artefactual.
MIN_DOMAIN_LEN = 110
short = f.VHH_domain.str.len() < MIN_DOMAIN_LEN
print(f"truncated domains dropped (<{MIN_DOMAIN_LEN} aa): "
      f"{f.loc[short, 'VHH_domain'].nunique()} domains, {int(short.sum())} rows")
f = f[~short]

# Drop any (domain, antigen) assayed with contradictory labels.
bad = (f.groupby(["VHH_domain", "Ag_label"]).label.nunique().loc[lambda s: s > 1])
print(f"(domain, antigen) combinations with contradictory labels: {len(bad):,}")
if len(bad):
    key = set(bad.index)
    f = f[~pd.MultiIndex.from_arrays([f.VHH_domain, f.Ag_label]).isin(key)]

tab = f.pivot_table(index="VHH_domain", columns="Ag_label",
                    values="label", aggfunc="max")

rows = []
for m in MUTANTS:
    both = tab[[WT, m]].dropna()
    for dom_, (w, mu) in both.iterrows():
        if w != mu:
            rows.append(dict(VHH_domain=dom_, mutant=m,
                             direction="lose" if w == 1 else "gain",
                             label_WT=int(w), label_mutant=int(mu)))
flips = pd.DataFrame(rows)

# Join the section-3 clonotypes. Independence is counted on these, not on rows.
flips = flips.merge(clo[["VHH_domain", "clonotype", "cdr3"]],
                    on="VHH_domain", how="left", validate="many_to_one")
assert flips.clonotype.notna().all(), "flip VHH missing from the clonotype table"

print(f"\nflip pairs {len(flips)}   distinct VHH domains {flips.VHH_domain.nunique()}"
      f"   distinct clonotypes {flips.clonotype.nunique()}")

summary = (flips.groupby(["mutant", "direction"])
                .agg(pairs=("VHH_domain", "size"),
                     domains=("VHH_domain", "nunique"),
                     clonotypes=("clonotype", "nunique")))
print("\n" + summary.to_string())

loss = flips[flips.direction == "lose"]
print(f"\nLOSS direction: {len(loss)} pairs -> {loss.VHH_domain.nunique()} domains "
      f"-> {loss.clonotype.nunique()} INDEPENDENT CLONOTYPES")
print("\nclonotypes available per mutant (loss):")
for m in MUTANTS:
    cs = sorted(int(x) for x in loss.loc[loss.mutant == m, "clonotype"].unique())
    print(f"  {m:<12} ({len(cs):>2}) {cs}")

flips.to_csv(PRO / "flip_pairs.csv", index=False)
print(f"\nwrote {PRO / 'flip_pairs.csv'}  ({len(flips)} rows)")

# ---------------------------------------------------------------------------
# Collapsed view: one row per (clonotype, mutant, direction) — the unit a panel
# is selected on, since pairs within a clonotype are one lineage sampled twice.
# The representative VHH is the lexicographically first domain in the clonotype:
# arbitrary but deterministic. There are no read counts in the release, so there
# is no "most abundant member" to prefer.
# ONE representative per clonotype, not per (clonotype, mutant): the member
# assayed in the most of that clonotype's (mutant, direction) rows, ties broken
# lexicographically so the choice is deterministic. Picking per-mutant would hand
# the same lineage different sequences for different mutants for no reason.
rep_of = {}
for c_, g in flips.groupby("clonotype"):
    rows = set(zip(g.mutant, g.direction))
    cover = {d: set(zip(gg.mutant, gg.direction)) for d, gg in g.groupby("VHH_domain")}
    best = max(sorted(cover), key=lambda d: len(cover[d]))
    # The representative must itself have been assayed in every row it is used
    # for, otherwise we would assert a flip that was never measured.
    assert cover[best] == rows, f"clonotype {c_}: no member covers all its rows"
    rep_of[c_] = best

cdr3_of = dict(zip(flips.VHH_domain, flips.cdr3))
by_clono = (flips.groupby(["clonotype", "mutant", "direction"])
                 .agg(n_pairs=("VHH_domain", "size"),
                      n_domains=("VHH_domain", "nunique"))
                 .reset_index())
by_clono["rep_VHH_domain"] = by_clono.clonotype.map(rep_of)
by_clono["rep_cdr3"] = by_clono.rep_VHH_domain.map(cdr3_of)
by_clono = by_clono.sort_values(["direction", "mutant", "clonotype"])
assert by_clono.rep_VHH_domain.nunique() == flips.clonotype.nunique(), \
    "one representative per clonotype expected"

by_clono.to_csv(PRO / "flip_pairs_clonotypes.csv", index=False)
print(f"wrote {PRO / 'flip_pairs_clonotypes.csv'}  ({len(by_clono)} rows)")

print(f"\ncollapsed: {len(flips)} pairs -> {len(by_clono)} (clonotype, mutant, direction) rows")
by_clono

truncated domains dropped (<110 aa): 99 domains, 217 rows
(domain, antigen) combinations with contradictory labels: 3



flip pairs 34   distinct VHH domains 19   distinct clonotypes 17

                      pairs  domains  clonotypes
mutant     direction                            
IL-6_E138A gain           4        4           3
           lose           8        8           8
IL-6_F102A gain           2        2           2
           lose          10       10           9
IL-6_F153A gain           2        2           2
           lose           8        8           8

LOSS direction: 26 pairs -> 12 domains -> 11 INDEPENDENT CLONOTYPES

clonotypes available per mutant (loss):
  IL-6_F102A   ( 9) [25, 30, 50, 53, 101, 110, 201, 490, 12476]
  IL-6_E138A   ( 8) [25, 30, 50, 53, 83, 201, 505, 12476]
  IL-6_F153A   ( 8) [30, 50, 53, 83, 110, 201, 505, 12476]

wrote /Users/roessner/Documents/PostDoc/Data/261003_MD_single_interface_mutations/data/processed/flip_pairs.csv  (34 rows)
wrote /Users/roessner/Documents/PostDoc/Data/261003_MD_single_interface_mutations/data/processed/flip_pairs_clonotypes.csv  (3

,clonotype,mutant,direction,n_pairs,n_domains,rep_VHH_domain,rep_cdr3
0,1,IL-6_E138A,gain,2,2,QVQLQESGGGLVQPGGSLRLSCAASGFTLDYYNIGWFRQAPGKERK...,AADRCPDPCSLATTPLSGREYDY
19,145,IL-6_E138A,gain,1,1,QVQLQESGGGSVQVGGSLSLSCVVSGRAISNYAMGWFRQALGAERE...,ASTRRGIMREEYYAY
27,1226,IL-6_E138A,gain,1,1,QVQLQESGGGLVQAGGSLRLSCAASGRTLSSYAMGWFRQAPGKERE...,AADRVYAVATMFSEY
13,82,IL-6_F102A,gain,1,1,QVQLQESGGGLVQAGDSLRLSCAASGRTFSSYNMGWFRQAPGKERE...,ARQLGGSYYIPRGYDS
28,4221,IL-6_F102A,gain,1,1,QVQLQESGGGLVQAGGSLRLSCAASGSTVSFNTMGWYRQAPGKQRE...,RFTDWSGYDY
1,1,IL-6_F153A,gain,1,1,QVQLQESGGGLVQPGGSLRLSCAASGFTLDYYNIGWFRQAPGKERK...,AADRCPDPCSLATTPLSGREYDY
26,941,IL-6_F153A,gain,1,1,QVQLQESGGGLAQPGGSLRLSCTASGNIFSINAMGWHRQAPGEQRN...,LAGRLGYKDGELTAENVS
2,25,IL-6_E138A,lose,1,1,QVQLQESGGGLVQAGGSLRLSCAGSGVTFSIYTMGWFRQAPGKERE...,AAKLLARTEYDY
4,30,IL-6_E138A,lose,1,1,QVQLQESGGGLVQAGGSLRLSCAASGRTFSSYTMGWFRRAPGKERE...,AALPTYSSYYTEADSAKYDY
7,50,IL-6_E138A,lose,1,1,QVQLQESGGGLVQPGGSLRLSCAASGIIFSINDMGWYRQVPGKQRE...,NAKSLLRGPY


## 5. Per-target design files

One `{target}.csv` per antigen in the project root, columns `Design` and
`Sequence`, built from `flip_pairs_clonotypes.csv`.

`Sequence` is the **mature VHH domain** — 22-residue phagemid signal peptide and
C-terminal His6 already removed. These are the sequences to fold / dock /
simulate; they start at `QVQL...`.

**Which VHHs land in which file.** Every flip pair needs *two* complexes, so a
VHH that flips on `IL-6_F102A` must be modelled against both `IL-6_F102A` and
`IL-6_WTs`:

* `IL-6_WTs.csv` — the union of all representatives (every pair has a WT side)
* `IL-6_<mut>.csv` — the representatives that flip on that mutant

**`Design` IDs are global, not per-file.** `VHH_07` is the same sequence in every
file, so the per-target outputs join cleanly downstream. IDs are assigned over
the sorted unique sequences, so they are stable across re-runs.

> Note: there are slightly more unique sequences than clonotypes. Clonotype 50
> contributes two members — the sibling that flips on `F102A` is not the one that
> flips on `E138A`/`F153A`. Each row keeps the VHH *actually measured* in that
> pair; forcing one representative per clonotype would assert flips that were
> never assayed.

In [6]:
by_clono = pd.read_csv(PRO / "flip_pairs_clonotypes.csv")

# Global, stable Design IDs over the sorted unique mature sequences.
seqs = sorted(by_clono.rep_VHH_domain.unique())
width = len(str(len(seqs)))
design = {s: f"VHH_{i:0{width}d}" for i, s in enumerate(seqs, start=1)}
print(f"unique mature VHH sequences: {len(seqs)}  ->  "
      f"{design[seqs[0]]} .. {design[seqs[-1]]}")

written = {}
for target in TARGETS:
    if target == WT:
        sel = seqs                                   # every pair has a WT side
    else:
        sel = sorted(by_clono.loc[by_clono.mutant == target, "rep_VHH_domain"].unique())
    out = (pd.DataFrame({"Design": [design[s] for s in sel], "Sequence": sel})
             .sort_values("Design").reset_index(drop=True))
    out.to_csv(WD / f"{target}.csv", index=False)
    written[target] = out
    print(f"  {target}.csv{'':<4} {len(out):>3} sequences")

# A mutant sequence absent from the WT file means an incomplete pair.
wt_set = set(written[WT].Sequence)
for t, df_ in written.items():
    if t != WT:
        missing = set(df_.Sequence) - wt_set
        assert not missing, f"{t}: {len(missing)} sequences absent from {WT}.csv"
assert all(s.startswith("QVQL") for s in seqs), "sequence does not start at the domain"
assert not any("HHHHHH" in s for s in seqs), "residual His tag"
print("\nchecks passed: every mutant sequence present in the WT file, "
      "no wrapper residue left")
print(f"length range: {min(map(len, seqs))}-{max(map(len, seqs))} aa")
written[WT].head()

unique mature VHH sequences: 17  ->  VHH_01 .. VHH_17
  IL-6_WTs.csv      17 sequences
  IL-6_F102A.csv      11 sequences
  IL-6_E138A.csv      11 sequences
  IL-6_F153A.csv      10 sequences

checks passed: every mutant sequence present in the WT file, no wrapper residue left
length range: 116-130 aa


,Design,Sequence
0,VHH_01,QVQLQESGGGLAQPGGSLRLSCTASGNIFSINAMGWHRQAPGEQRN...
1,VHH_02,QVQLQESGGGLVQAGDSLRLSCAASGRTFSSYNMGWFRQAPGKERE...
2,VHH_03,QVQLQESGGGLVQAGGSLKLSCAASGSSESNYAMGWFRQAPGKERE...
3,VHH_04,QVQLQESGGGLVQAGGSLRLSCAASGRTFNSYDLGWFRQAPGKERE...
4,VHH_05,QVQLQESGGGLVQAGGSLRLSCAASGRTFSGYAMGWFRQAPGKERE...
